# 🎵 Stable Audio 3 Small — ONNX Export on Colab T4

This notebook:
1. Checks the T4 GPU is available
2. Mounts Google Drive (to save ONNX files permanently)
3. Clones your GitHub repo
4. Installs all dependencies
5. Logs in to HuggingFace (SA3 may be a gated model)
6. Exports all 4 modules (text_encoder, conditioner, DiT, decoder) to ONNX FP16
7. Runs INT8 quantization
8. Validates all exports
9. Zips and saves the models to Google Drive

**Runtime → Change runtime type → T4 GPU** (free tier is fine)

---
⏱ **Expected total time:** ~20–40 min (mostly model download + DiT export)

## Cell 1 — Check GPU

In [ ]:
import subprocess, sys

# Verify T4 GPU is attached
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    cc = torch.cuda.get_device_capability()
    tier = 'gpu_high' if cc[0] >= 7 else 'gpu_low'
    print(f'Compute cap     : SM {cc[0]}.{cc[1]} → tier = {tier}')
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
    sys.exit(1)

## Cell 2 — Mount Google Drive (saves ONNX files permanently)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/stable_audio_onnx_models'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Drive output dir: {DRIVE_OUTPUT}')

## Cell 3 — Clone your GitHub repo

**Edit `GITHUB_REPO` below to your repo URL before running.**

In [ ]:
# ✏️  EDIT THIS — your GitHub repo URL
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git'

# Clone into /content/stable_audio_onnx
!git clone {GITHUB_REPO} /content/repo

# The package lives inside the repo under stable_audio_onnx/
import os, sys
PACKAGE_DIR = '/content/repo/stable_audio_onnx'
sys.path.insert(0, PACKAGE_DIR)
os.chdir(PACKAGE_DIR)
print(f'Working dir: {os.getcwd()}')
print(os.listdir('.'))

## Cell 4 — Install dependencies

Colab already has PyTorch + CUDA, so we only need the extra packages.
This takes ~3–5 minutes.

In [ ]:
# HuggingFace ecosystem (pinned to match our export scripts)
!pip install -q transformers==4.40.0 diffusers==0.27.2 accelerate==0.29.3

# Stability AI stable-audio-tools
!pip install -q stable-audio-tools

# ONNX core + Runtime (GPU build for CUDA)
!pip install -q onnx==1.16.0 onnxruntime-gpu==1.18.0

# FP16 conversion helper (for F4 NaN recovery)
!pip install -q onnxconverter-common>=1.13.0

# Audio + validation
!pip install -q soundfile scipy

print('\n✅ All dependencies installed.')

## Cell 5 — HuggingFace login

Stable Audio 3 Small may be a **gated model** requiring HuggingFace account access.

1. Go to https://huggingface.co/stabilityai/stable-audio-3-small
2. Click **Agree and access repository** (if prompted)
3. Get your token at https://huggingface.co/settings/tokens
4. Paste it below (or use `HF_TOKEN` secret in Colab Secrets panel)

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Try Colab Secrets first (recommended — avoids pasting token in notebook)
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('✅ Logged in via Colab Secrets (HF_TOKEN)')
except Exception:
    # Manual fallback
    print('Colab Secret HF_TOKEN not found — falling back to interactive login.')
    print('Tip: Add your token via Colab Secrets (🔑 icon in left panel) as HF_TOKEN')
    login()   # opens interactive prompt

# Verify the model is accessible
from huggingface_hub import model_info
for mid in ['stabilityai/stable-audio-3-small', 'stabilityai/stable-audio-3-small-sfx']:
    try:
        info = model_info(mid)
        print(f'✅ {mid} — accessible ({info.modelId})')
    except Exception as e:
        print(f'⚠️  {mid} — {e}')
        print(f'   Update MODEL_IDS in config.py if the ID has changed.')

## Cell 6 — Verify device_selector sees the T4 GPU

In [ ]:
import onnxruntime as ort
from device_selector import get_hardware_tier

print('Available ORT providers:', ort.get_available_providers())
tier = get_hardware_tier()
print(f'Hardware tier    : {tier}')

from config import DIFFUSION_STEPS, TOLERANCE
print(f'Diffusion steps  : {DIFFUSION_STEPS[tier]}')
print(f'FP16 tolerance   : {TOLERANCE["fp16"]}')

## Cell 7 — Export all 4 modules to ONNX FP16

This is the main event. Order: text_encoder → conditioner → DiT → decoder

**DiT export is the slowest step** (~5–15 min depending on model size).
Watch the progress logs — each module prints its output path when done.

In [ ]:
# Export Music variant
from export.export_all import run as export_run

print('=' * 60)
print('EXPORTING: music variant')
print('=' * 60)
music_paths = export_run('music')

In [ ]:
# Export SFX variant
print('=' * 60)
print('EXPORTING: sfx variant')
print('=' * 60)
sfx_paths = export_run('sfx')

## Cell 8 — INT8 Quantization (all 4 modules)

Produces INT8 versions in `models/<variant>/int8/`. 
Recommended for all hardware — smaller files, faster on CPU.

In [ ]:
from quantization.int8_quantize import quantize_int8

for variant in ['music', 'sfx']:
    print(f'\nQuantizing {variant} → INT8 ...')
    quantize_int8(variant)

print('\n✅ INT8 quantization complete.')

## Cell 9 — INT4 Quantization (DiT + decoder only) — Optional

Smallest file size. Skip if you only need INT8.

In [ ]:
from quantization.int4_quantize import quantize_int4

for variant in ['music', 'sfx']:
    print(f'\nQuantizing {variant} → INT4 ...')
    quantize_int4(variant)

print('\n✅ INT4 quantization complete.')

## Cell 10 — Validate all exports

In [ ]:
from validate.validate_single import validate_module
from pathlib import Path
from config import OUTPUT_ROOT

modules  = ['text_encoder', 'conditioner', 'dit', 'decoder']
variants = ['music', 'sfx']

all_passed = True
for variant in variants:
    for name in modules:
        path = Path(OUTPUT_ROOT) / variant / 'fp16' / f'{name}.onnx'
        if path.exists():
            try:
                validate_module(name, variant, str(path), dtype_key='fp16')
            except Exception as e:
                print(f'❌ {variant}/{name}: {e}')
                all_passed = False
        else:
            print(f'⚠️  {variant}/{name}: file not found — was export successful?')
            all_passed = False

if all_passed:
    print('\n✅ All modules validated — no NaN detected.')
else:
    print('\n⚠️  Some validations failed. See messages above.')

## Cell 11 — Check file sizes

In [ ]:
from pathlib import Path
from config import OUTPUT_ROOT

total_bytes = 0
print(f'{"File":<55} {"Size":>10}')
print('-' * 67)
for onnx_file in sorted(Path(OUTPUT_ROOT).rglob('*.onnx')):
    size = onnx_file.stat().st_size
    total_bytes += size
    rel = str(onnx_file.relative_to(OUTPUT_ROOT))
    print(f'{rel:<55} {size/1e6:>9.1f} MB')
print('-' * 67)
print(f'{"TOTAL":<55} {total_bytes/1e9:>9.2f} GB')

## Cell 12 — Quick inference test (SFX)

Generate a short sound effect to verify the pipeline works end-to-end.

In [ ]:
from inference.run_inference import run_onnx_pipeline, save_wav
import IPython.display as ipd
import numpy as np

PROMPT   = 'A short rain shower on a tin roof'
DURATION = 3.0
OUT_WAV  = '/content/test_sfx.wav'

print(f'Generating: "{PROMPT}" ({DURATION}s) ...')
waveform = run_onnx_pipeline(
    prompt=PROMPT,
    duration_seconds=DURATION,
    variant='sfx',
    seed=42,
)

save_wav(waveform, OUT_WAV)

# Play inline in Colab
from config import SAMPLE_RATE
ipd.display(ipd.Audio(waveform, rate=SAMPLE_RATE))
print(f'\n✅ SFX inference complete — {len(waveform)/SAMPLE_RATE:.2f}s of audio generated.')

## Cell 13 — Save models to Google Drive

Zips the entire `models/` folder and copies it to your Drive.
Then you can download it locally whenever you want.

In [ ]:
import shutil, os
from pathlib import Path
from config import OUTPUT_ROOT

DRIVE_OUTPUT = '/content/drive/MyDrive/stable_audio_onnx_models'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

print('Zipping models/ folder ...')
zip_path = shutil.make_archive(
    base_name=f'{DRIVE_OUTPUT}/stable_audio_onnx_models',
    format='zip',
    root_dir='.',
    base_dir=OUTPUT_ROOT,
)
size_gb = Path(zip_path).stat().st_size / 1e9
print(f'✅ Saved to Drive: {zip_path}  ({size_gb:.2f} GB)')
print('\nYou can now download this zip from Google Drive.')

## Cell 14 — (Optional) Download models directly to your browser

Alternative to Drive — downloads individual files directly.
Useful for grabbing just one module (e.g., only the INT8 DiT).

In [ ]:
from google.colab import files
from pathlib import Path
from config import OUTPUT_ROOT

# Change these to select what to download
DOWNLOAD_VARIANT   = 'sfx'     # 'music' or 'sfx'
DOWNLOAD_PRECISION = 'int8'    # 'fp16', 'int8', or 'int4'

target_dir = Path(OUTPUT_ROOT) / DOWNLOAD_VARIANT / DOWNLOAD_PRECISION
for onnx_file in sorted(target_dir.glob('*.onnx')):
    print(f'Downloading: {onnx_file.name} ({onnx_file.stat().st_size/1e6:.1f} MB)')
    files.download(str(onnx_file))

---
## ✅ Done!

Your ONNX models are saved to Google Drive at:
`My Drive / stable_audio_onnx_models / stable_audio_onnx_models.zip`

### Output layout inside the zip:
```
models/
├── music/
│   ├── fp16/  text_encoder.onnx  dit.onnx  decoder.onnx  conditioner.onnx
│   ├── int8/  text_encoder.onnx  dit.onnx  decoder.onnx  conditioner.onnx
│   └── int4/  dit.onnx  decoder.onnx
└── sfx/
    ├── fp16/  text_encoder.onnx  dit.onnx  decoder.onnx  conditioner.onnx
    ├── int8/  text_encoder.onnx  dit.onnx  decoder.onnx  conditioner.onnx
    └── int4/  dit.onnx  decoder.onnx
```

### To run inference locally after downloading:
```bash
cd stable_audio_onnx
python -m inference.run_inference \
    --prompt "Rolling thunder" \
    --duration 3.0 \
    --variant sfx \
    --out thunder.wav
```